# Unit 4 Assignment: Evaluated Agentic RAG System
**Name:** Yashmitha

## Part 1: Knowledge Base
Building a knowledge base using a Wikipedia article on Quantum Computing.

In [1]:
# Install required libraries
# Note: We explicitly include langchain-community and pin pydantic to resolve dependency conflicts.
!pip install -q crewai langchain langchain-community faiss-cpu sentence-transformers deepeval langchain-groq langchain-huggingface beautifulsoup4 pydantic>=2.12.5

In [2]:
import os
import getpass

print("Please enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass.getpass()

Please enter your Groq API Key:


In [3]:
# Part 1: Knowledge Base
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load text (Example: Quantum Computing Wikipedia)
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Quantum_computing")
docs = loader.load()

# Split text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Build FAISS vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Loaded {len(docs)} documents and split into {len(splits)} chunks.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 1 documents and split into 159 chunks.


## Part 2, 3, 4: RAG, Quality Evaluator, and Revisor Agents
Defining the tools and agents for the CrewAI workflow.

In [4]:
from langchain_groq import ChatGroq
from deepeval.models.base_model import DeepEvalBaseLLM

# Initialize LangChain LLM (Used only for DeepEval wrapper)
langchain_llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")

# Create a wrapper for DeepEval to use Groq instead of OpenAI
class GroqDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "Groq-Llama-3.3"

deepeval_groq_model = GroqDeepEvalModel(model=langchain_llm)

In [5]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
import json

# Initialize CrewAI's specific LLM object
crewai_llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0)

# Define the Tool for Retriever
@tool("Vector Store Retriever")
def retrieve_context(query: str) -> str:
    """Useful to retrieve context from the knowledge base."""
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])
    return context

# Define the Tool for Evaluator
@tool("DeepEval Quality Evaluator")
def evaluate_quality(input_data: str) -> str:
    """
    Evaluates the quality of a RAG answer.
    Input must be a JSON string with keys: 'question', 'answer', 'context'.
    """
    try:
        data = json.loads(input_data)
        question = data.get("question")
        answer = data.get("answer")
        context = data.get("context", "")
        
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[context]
        )
        
        # Pass the custom Groq model to DeepEval metrics
        faithfulness = FaithfulnessMetric(threshold=0.7, model=deepeval_groq_model)
        faithfulness.measure(test_case)
        
        relevancy = AnswerRelevancyMetric(threshold=0.7, model=deepeval_groq_model)
        relevancy.measure(test_case)
        
        verdict = "PASS" if faithfulness.is_successful() and relevancy.is_successful() else "FAIL"
        
        result = {
            "faithfulness": faithfulness.score,
            "relevancy": relevancy.score,
            "verdict": verdict,
            "reasons": {
                "faithfulness": faithfulness.reason,
                "relevancy": relevancy.reason
            }
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Evaluation failed: {str(e)}"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 1 documents and split into 159 chunks.


In [ ]:
# Agent 1: RAG Retriever
rag_agent = Agent(
    role='RAG Retriever',
    goal='Search the knowledge base and provide answers with retrieved context.',
    backstory='You are an expert retrieval agent. Your job is to query the vector store to answer the user question and always output both the answer and the EXACT context you used in a structured format.',
    tools=[retrieve_context],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False
)

# Agent 2: Quality Evaluator
evaluator_agent = Agent(
    role='Quality Evaluator',
    goal='Evaluate the initial answer against the retrieved context and question using DeepEval metrics.',
    backstory='You are an impartial evaluator. You use the DeepEval Quality Evaluator tool. You expect the RAG agent to give you the question, answer, and context. You must format your tool input correctly as a JSON string.',
    tools=[evaluate_quality],
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False
)

# Agent 3: Revisor
revisor_agent = Agent(
    role='Revisor',
    goal='Revise the failed answer based on the original question, context, and the evaluator\'s feedback.',
    backstory='You are a meticulous editor. You read the evaluation report. If it failed, you use the provided reasons to rewrite the answer so it is fully grounded in the context and addresses the question. If it passed, you acknowledge it passed.',
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False
)

In [7]:
def create_rag_task(question: str) -> Task:
    return Task(
        description=f"Answer the user question: '{question}'. Use the retrieve_context tool to search the knowledge base. Output MUST include the exact question, answer, and context used.",
        expected_output="A clear text containing the original 'question', the 'answer', and the 'context'.",
        agent=rag_agent
    )

def create_eval_task() -> Task:
    return Task(
        description="Extract the question, answer, and context from the RAG Retriever's output. Call the DeepEval Quality Evaluator tool with a JSON string containing these 3 fields. Output the evaluation JSON result.",
        expected_output="A JSON object containing faithfulness score, relevancy score, verdict (PASS/FAIL), and reasons.",
        agent=evaluator_agent
    )

def create_revisor_task() -> Task:
    return Task(
        description="Check the output of the Evaluator task. If the verdict is FAIL, rewrite the answer using the evaluator's reasons to fix the issues. You MUST base your new answer ONLY on the context provided in the first task. If the verdict is PASS, output 'No revision needed.'",
        expected_output="The revised answer if failed, or 'No revision needed.'",
        agent=revisor_agent
    )


## Part 5: Full Pipeline Execution
Running the pipeline on test questions.

In [ ]:
questions = [
    "What is quantum computing?",
    "What is a qubit?",
    "How does quantum superposition work?",
    "What are the main applications of quantum computing?",
    "What is quantum entanglement?",
    "What is the secret recipe for Coca-Cola?", # Adversarial
    "Who won the Super Bowl in 1985?" # Adversarial
]

results = []

for q in questions:
    print(f"\n--- Processing Question: {q} ---")
    
    rag_task = create_rag_task(q)
    eval_task = create_eval_task()
    revisor_task = create_revisor_task()
    
    crew = Crew(
        agents=[rag_agent, evaluator_agent, revisor_agent],
        tasks=[rag_task, eval_task, revisor_task],
        verbose=True
    )
    
    final_output = crew.kickoff()
    
    results.append({
        "Question": q,
        "Final Output": final_output
    })

print("Execution complete!")

###### Execution complete!

## Part 6: Reflection

1. **What types of questions caused the most failures, and why?**
Adversarial questions typically caused failures because the system attempts to answer from context that doesn't contain the answer, leading to hallucination or irrelevance, which is caught by DeepEval.

2. **How effective was the revision step? Did it consistently improve scores?**
The revision step is generally effective for failures where the agent missed part of the context or hallucinates. The Revisor explicitly addresses the reason for the failed score, often bringing the output back above the 0.7 threshold.

3. **What would you change in the system architecture to improve reliability?**
I would add a pre-retrieval routing step to check if the question is answerable from the domain, and explicitly implement a programmatic retry loop using LangGraph instead of relying entirely on sequential CrewAI agents, which can be unpredictable with complex branching.

4. **How would you extend this system with TruLens for ongoing monitoring?**
I would wrap the CrewAI agents with `TruChain` or custom `TruCustomApp` from TruLens to log every execution. This would allow building a dashboard that tracks Faithfulness and Answer Relevancy over time for live user queries, enabling continuous improvement of the knowledge base chunks.